In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectPercentile, chi2
import umap.umap_ as umap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.metrics import balanced_accuracy_score, f1_score
from calidad_pqrs.utils import load_directory, load_data, drop_data, mapping_data, clean_text_TfIdf, save_model, build_predictions_dataframe, load_model, optimize_threshold
from calidad_pqrs.config import MODEL_PROCESS_DIR
import optuna

c:\Users\luisarmz\OneDrive - Seguros Suramericana, S.A\Documentos\2025\S2\Calidad PQRS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Peprocesamiento

In [2]:
paths = load_directory(directory='Train')

In [3]:
data = load_data(paths)

Leyendo las descripciones de las quejas...


In [4]:
data = drop_data(data)
data = mapping_data(data)

Homologando procesos y causas con poca participación...


In [5]:
data_procesada = clean_text_TfIdf(data)

### Representación visual de las clases

In [ ]:
# tfidf = TfidfVectorizer(decode_error='ignore')
# X_tfidf = tfidf.fit_transform(data_procesada['Descripción_TfIdf'])

# umap_model = umap.UMAP(n_components=2, random_state=666, n_neighbors=25, min_dist=0.27)
# X_umap = umap_model.fit_transform(X_tfidf)

# y = data_procesada['Proceso']
# y_codes = y.astype("category").cat.codes
# cmap = cm.get_cmap('Paired', len(np.unique(y_codes)))


# plt.figure(figsize=(10, 7))
# scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_codes, cmap=cmap, alpha=0.8)
# plt.xlabel("UMAP 1")
# plt.ylabel("UMAP 2")
# plt.title("Visualización con UMAP sobre TF-IDF")
# handles, _ = scatter.legend_elements()
# plt.legend(handles, y.astype("category").cat.categories, title="Clases")

# plt.show()

In [ ]:
# display(
#     data_procesada.shape,
#     X_tfidf.shape
# )

### clasificación con regresión logística

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    data_procesada['Descripción_TfIdf'],
    data_procesada['Proceso'],
    test_size = 0.2,
    shuffle = True,
    random_state= 666,
)

In [ ]:
display(
    len(y_train.value_counts(normalize=True)),
    len(y_test.value_counts(normalize=True))
)

In [7]:
def objective(trial):

    #percentile = trial.suggest_float('percentile', 40, 100)
    C = trial.suggest_float('C', 1e-1, 1e1, log=True)
    min_df = trial.suggest_float('min_df', 1e-5, 9e-4)

    pipe = Pipeline(
        [
            ('preprocessor', TfidfVectorizer(decode_error='ignore', min_df=min_df, max_df=0.8)),
            #('select', SelectPercentile(score_func=chi2, percentile=percentile)),
            ('classifier', LogisticRegression(
                random_state=666,
                max_iter=6000,
                class_weight='balanced',
                penalty='l2',
                solver='liblinear',
                C=C
            )),
        ]
    )

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=666)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)

    return scores.mean()


study = optuna.create_study(direction="maximize", study_name='logistic_regression_optimization')
study.optimize(objective, n_trials=66)
best_params = study.best_params

[I 2026-02-12 10:46:06,057] A new study created in memory with name: logistic_regression_optimization
[I 2026-02-12 10:46:20,033] Trial 0 finished with value: 0.6781879834409315 and parameters: {'C': 0.4138293080798262, 'min_df': 0.0007000891202880178}. Best is trial 0 with value: 0.6781879834409315.
[I 2026-02-12 10:46:33,787] Trial 1 finished with value: 0.693146500614086 and parameters: {'C': 5.500932618205939, 'min_df': 0.0005581213870912195}. Best is trial 1 with value: 0.693146500614086.
[I 2026-02-12 10:46:46,885] Trial 2 finished with value: 0.6824412306437553 and parameters: {'C': 0.5903018265298295, 'min_df': 0.00010035920579506674}. Best is trial 1 with value: 0.693146500614086.
[I 2026-02-12 10:46:52,343] Trial 3 finished with value: 0.6928364673302561 and parameters: {'C': 4.299177303818615, 'min_df': 0.0006876277419738424}. Best is trial 1 with value: 0.693146500614086.
[I 2026-02-12 10:46:58,718] Trial 4 finished with value: 0.7015475012177725 and parameters: {'C': 4.937

In [8]:
best_params

{'C': 2.99196241804849, 'min_df': 4.0688294837300544e-05}

In [11]:
model_process = Pipeline(
    [
        ('preprocessor', TfidfVectorizer(decode_error='ignore', min_df=best_params['min_df'], max_df=0.8)),
        #('select', SelectPercentile(score_func=chi2, percentile=best_params['percentile'])),
        ('classifier', LogisticRegression(random_state=666, max_iter=6000, class_weight='balanced', penalty='l2', solver='liblinear', C=best_params['C'])),
    ]
)

model_process.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 TfidfVectorizer(decode_error='ignore', max_df=0.8,
                                 min_df=4.0688294837300544e-05)),
                ('classifier',
                 LogisticRegression(C=2.99196241804849, class_weight='balanced',
                                    max_iter=6000, random_state=666,
                                    solver='liblinear'))])

In [12]:
y_pred_train = model_process.predict(X_train)
y_pred_test = model_process.predict(X_test)

print('balanced accuracy train:', balanced_accuracy_score(y_train, y_pred_train))
print('balanced accuracy test:', balanced_accuracy_score(y_test, y_pred_test))
print('f1 score train (weighted):', f1_score(y_train, y_pred_train, average='weighted'))
print('f1 score test (weighted):', f1_score(y_test, y_pred_test, average='weighted'))
print('f1 score train (macro):', f1_score(y_train, y_pred_train, average='macro'))
print('f1 score test (macro):', f1_score(y_test, y_pred_test, average='macro'))

print('\nscore train:', model_process.score(X_train, y_train))
print('score test:', model_process.score(X_test, y_test))

balanced accuracy train: 0.9332978808790429
balanced accuracy test: 0.6956876156447822
f1 score train (weighted): 0.9266759132707268
f1 score test (weighted): 0.8146827484308996
f1 score train (macro): 0.9132010159996764
f1 score test (macro): 0.7062901201025462

score train: 0.9263439101515647
score test: 0.8144352376798953


In [13]:
def matriz(yt, yp):
    labels = np.unique(yt)
    matrix = confusion_matrix(y_true=yt, y_pred=yp, labels=labels, normalize='pred')
    
    index = [f"{label} (Clase Real)" for label in labels]
    columns = [f"{label} (Predicción)" for label in labels]
    
    return pd.DataFrame(matrix, index=index, columns=columns)


matrix_test = matriz(y_test, model_process.predict(X_test))
display(matrix_test)

,ASESORIA Y VENTA (Predicción),ASISTENCIA SALUD (Predicción),EVALUACION (Predicción),EXPEDICION (Predicción),RECAUDOS (Predicción),RECLAMACIONES SALUD (Predicción),RELACIONAMIENTO Y SERVICIO AL CLIENTE (Predicción),TRANSFORMACION DIGITAL (Predicción)
ASESORIA Y VENTA (Clase Real),0.571429,0.022862,0.105263,0.167598,0.037838,0.039861,0.034682,0.006452
ASISTENCIA SALUD (Clase Real),0.173302,0.905794,0.052632,0.047486,0.016216,0.204506,0.144509,0.029032
EVALUACION (Clase Real),0.014052,0.000788,0.578947,0.016760,0.000000,0.003466,0.000000,0.000000
EXPEDICION (Clase Real),0.135831,0.004730,0.105263,0.631285,0.162162,0.015598,0.034682,0.009677
RECAUDOS (Clase Real),0.011710,0.001577,0.000000,0.072626,0.756757,0.001733,0.000000,0.006452
RECLAMACIONES SALUD (Clase Real),0.046838,0.039811,0.157895,0.027933,0.016216,0.715771,0.023121,0.022581
RELACIONAMIENTO Y SERVICIO AL CLIENTE (Clase Real),0.035129,0.019708,0.000000,0.019553,0.000000,0.005199,0.722543,0.029032
TRANSFORMACION DIGITAL (Clase Real),0.011710,0.004730,0.000000,0.016760,0.010811,0.013865,0.040462,0.896774


In [14]:
save_model(
    model = model_process,
    model_dir = MODEL_PROCESS_DIR,
    model_name = 'salud_process_classifier.pkl',
    X_test = X_test,
    y_test = y_test
)

Nuevo modelo guardado.
Score nuevo modelo: 0.7062901201025462
Score nuevo anterior: 0.676640566054042


In [15]:
tfidf_model = load_model(MODEL_PROCESS_DIR, 'salud_process_classifier.pkl')

In [16]:
y_pred_train = tfidf_model.predict(X_train)
y_pred_test = tfidf_model.predict(X_test)

print('balanced accuracy train:', balanced_accuracy_score(y_train, y_pred_train))
print('balanced accuracy test:', balanced_accuracy_score(y_test, y_pred_test))
print('f1 score train (weighted):', f1_score(y_train, y_pred_train, average='weighted'))
print('f1 score test (weighted):', f1_score(y_test, y_pred_test, average='weighted'))
print('f1 score train (macro):', f1_score(y_train, y_pred_train, average='macro'))
print('f1 score test (macro):', f1_score(y_test, y_pred_test, average='macro'))

print('\nscore train:', tfidf_model.score(X_train, y_train))
print('score test:', tfidf_model.score(X_test, y_test))

balanced accuracy train: 0.9332978808790429
balanced accuracy test: 0.6956876156447822
f1 score train (weighted): 0.9266759132707268
f1 score test (weighted): 0.8146827484308996
f1 score train (macro): 0.9132010159996764
f1 score test (macro): 0.7062901201025462

score train: 0.9263439101515647
score test: 0.8144352376798953


In [ ]:
predictions_dataframe = build_predictions_dataframe(
    model = tfidf_model,
    X_test = X_test,
    true_class_col='clase real',
    y_test = y_test,
    predicted_class_col='clase predicha',
    dataset_original = data_procesada,
    drop_cols = ('Proceso', 'Causa')
)

In [ ]:
thresholds = optimize_threshold(
    dataset = predictions_dataframe,
    threshold_error = 0.12
)

thresholds

In [ ]:
predictions_dataframe.to_excel('../Output/resultados_proceso.xlsx', index=False)